# Titanic Agent Tooling Benchmark — Combined Notebook

**MSIN0097 Predictive Analytics — Group Coursework 2025–26**

This notebook consolidates the code produced by all three agent tools (GitHub Copilot, Claude Code, and OpenAI Codex) across five benchmark tasks. It serves as the reproducible evidence base for the accompanying report. Each section presents the task specification, the code generated by each tool, and key observations.

**Tools evaluated:** GitHub Copilot (Chat panel, VS Code / Jupyter) · Claude Code (terminal-based agent) · OpenAI Codex (cloud-based agentic framework)

**Dataset:** Titanic survival dataset (891 passengers, 12 original features)

**Random seed:** `RANDOM_STATE = 42` used throughout

---
## Environment Setup

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Verify dataset is available
df_raw = pd.read_csv('data/Titanic-Dataset.csv')
print(f'Raw dataset shape: {df_raw.shape}')
print(f'Columns: {list(df_raw.columns)}')
print(f'\nMissing values:\n{df_raw.isnull().sum()[df_raw.isnull().sum() > 0]}')
print(f'\nTarget distribution:\n{df_raw["Survived"].value_counts()}')

---
## Task 1: Dataset Ingestion and Cleaning

**Specification:** Load `titanic.csv`, print a schema summary, clean the dataset (drop PassengerId/Name/Ticket/Cabin; impute Age with median; impute Embarked with mode; encode Sex as binary), print all transformations, and save as `titanic_clean.csv`.

**Success criteria:** Script runs end-to-end, all 4 transformations applied, zero missing values in output, shape = (891, 8).

### Copilot — Task 1

**Score: 4/12** — Used deprecated `inplace` pattern (`df['Age'].fillna(median_age, inplace=True)`) which is a silent no-op in pandas ≥ 2.0. Terminal log reported success, but the saved CSV retained 177 missing Age values and Embarked as a string column. This defect cascaded into Tasks 2 and 5.

In [ ]:
import pandas as pd

# Load the dataset
df = pd.read_csv('titanic.csv')

# Print schema summary
print("Schema Summary:")
for col in df.columns:
    print(f"{col}: {df[col].dtype}, missing: {df[col].isnull().sum()}")

# Clean the dataset
dropped_cols = ['PassengerId', 'Name', 'Ticket', 'Cabin']
df.drop(dropped_cols, axis=1, inplace=True)
print(f"Dropped columns: {', '.join(dropped_cols)}")

missing_age = df['Age'].isnull().sum()
median_age = df['Age'].median()
df['Age'].fillna(median_age, inplace=True)  # BUG: silent no-op in pandas >= 2.0
print(f"Imputed {missing_age} missing Age values with median: {median_age}")

missing_embarked = df['Embarked'].isnull().sum()
mode_embarked = df['Embarked'].mode()[0]
df['Embarked'].fillna(mode_embarked, inplace=True)
print(f"Imputed {missing_embarked} missing Embarked values with mode: '{mode_embarked}'")

df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})
print("Encoded Sex column as binary: male=0, female=1")

print(f"Final dataframe shape: {df.shape}")
df.to_csv('titanic_clean.csv', index=False)

### Claude Code — Task 1

**Score: 12/12** — Used `.fillna()` with reassignment (not `inplace`), avoiding the pandas ≥ 2.0 bug. All transformations logged, zero missing values in output.

In [ ]:
import pandas as pd

RANDOM_STATE = 42

# ─────────────────────────────────────────────
# 1. Load
# ─────────────────────────────────────────────
DATA_PATH = "../../data/Titanic-Dataset.csv"
OUTPUT_PATH = "titanic_clean.csv"

df = pd.read_csv(DATA_PATH)

# ─────────────────────────────────────────────
# 2. Schema summary
# ─────────────────────────────────────────────
print("=" * 55)
print("SCHEMA SUMMARY")
print("=" * 55)
summary = pd.DataFrame({
    "dtype": df.dtypes,
    "missing": df.isnull().sum(),
})
print(summary.to_string())
print(f"\nShape before cleaning: {df.shape}")

# ─────────────────────────────────────────────
# 3. Cleaning
# ─────────────────────────────────────────────
print("\n" + "=" * 55)
print("TRANSFORMATIONS")
print("=" * 55)

# 3a. Drop identifier / high-cardinality columns
drop_cols = ["PassengerId", "Name", "Ticket", "Cabin"]
df.drop(columns=drop_cols, inplace=True)
print(f"Dropped columns: {drop_cols}")

# 3b. Impute Age with median
age_missing = df["Age"].isnull().sum()
age_median = df["Age"].median()
df["Age"] = df["Age"].fillna(age_median)
print(f"Imputed {age_missing} missing Age values with median: {age_median}")

# 3c. Impute Embarked with mode
embarked_missing = df["Embarked"].isnull().sum()
embarked_mode = df["Embarked"].mode()[0]
df["Embarked"] = df["Embarked"].fillna(embarked_mode)
print(f"Imputed {embarked_missing} missing Embarked values with mode: '{embarked_mode}'")

# 3d. Encode Sex as binary (male=0, female=1)
sex_map = {"male": 0, "female": 1}
df["Sex"] = df["Sex"].map(sex_map)
print(f"Encoded Sex column as binary (male=0, female=1)")

# ─────────────────────────────────────────────
# 4. Final shape
# ─────────────────────────────────────────────
print("\n" + "=" * 55)
print("FINAL DATAFRAME")
print("=" * 55)
print(f"Shape after cleaning: {df.shape}")
print(f"\nRemaining missing values per column:")
print(df.isnull().sum().to_string())

# ─────────────────────────────────────────────
# 5. Save
# ─────────────────────────────────────────────
df.to_csv(OUTPUT_PATH, index=False)
print(f"\nSaved cleaned dataframe to: {OUTPUT_PATH}")


### Codex — Task 1

**Score: 12/12** — Production-grade: `main()` entry point, type hints, defensive column checks, transformation log list. Uses `.fillna()` with reassignment.

In [ ]:
from __future__ import annotations

import pandas as pd


RANDOM_STATE = 42
INPUT_PATH = "titanic.csv"
OUTPUT_PATH = "titanic_clean.csv"


def print_schema_summary(df: pd.DataFrame) -> None:
    print("Schema summary:")
    for column in df.columns:
        dtype = df[column].dtype
        missing = df[column].isna().sum()
        print(f"- {column}: dtype={dtype}, missing_values={missing}")


def main() -> None:
    df = pd.read_csv(INPUT_PATH)
    transformations: list[str] = []

    print_schema_summary(df)
    print()

    columns_to_drop = ["PassengerId", "Name", "Ticket", "Cabin"]
    existing_columns_to_drop = [column for column in columns_to_drop if column in df.columns]
    df = df.drop(columns=existing_columns_to_drop)
    transformations.append(
        "Dropped columns: " + ", ".join(existing_columns_to_drop)
    )

    missing_age_count = int(df["Age"].isna().sum())
    age_median = float(df["Age"].median())
    df["Age"] = df["Age"].fillna(age_median)
    transformations.append(
        f"Imputed {missing_age_count} missing Age values with median: {age_median}"
    )

    missing_embarked_count = int(df["Embarked"].isna().sum())
    embarked_mode = df["Embarked"].mode(dropna=True).iloc[0]
    df["Embarked"] = df["Embarked"].fillna(embarked_mode)
    transformations.append(
        f"Imputed {missing_embarked_count} missing Embarked values with mode: {embarked_mode}"
    )

    sex_mapping = {"male": 0, "female": 1}
    df["Sex"] = df["Sex"].map(sex_mapping)
    transformations.append("Encoded Sex column as binary: male=0, female=1")

    df.to_csv(OUTPUT_PATH, index=False)

    print("Transformations applied:")
    for transformation in transformations:
        print(f"- {transformation}")

    print()
    print(f"Final clean dataframe shape: {df.shape}")
    print(f"Saved cleaned dataframe to: {OUTPUT_PATH}")


if __name__ == "__main__":
    main()


---
## Task 2: Baseline Model Training + Evaluation Harness

**Specification:** Stratified 80/20 split, train LogisticRegression (max_iter=1000, random_state=42), report Accuracy/Precision/Recall/F1 (weighted) on test set, print confusion matrix. All preprocessing must occur after the split or within a Pipeline.

**Success criteria:** Code runs without error, metrics on test set only, no pre-split preprocessing, metrics in plausible range (~0.78–0.82 accuracy).

### Copilot — Task 2

**Score: 6/12** — Structurally correct code but crashed at execution because it inherited Copilot's corrupt T1 CSV (Embarked still a string → `ValueError: could not convert string to float: 'S'`). No preprocessing Pipeline used. No metrics produced.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

df = pd.read_csv('titanic_clean.csv')
X = df.drop('Survived', axis=1)
y = df['Survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)  # CRASHES: ValueError - Embarked column is still a string

y_pred = model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision (weighted): {precision_score(y_test, y_pred, average='weighted'):.4f}")
print(f"Recall (weighted): {recall_score(y_test, y_pred, average='weighted'):.4f}")
print(f"F1 Score (weighted): {f1_score(y_test, y_pred, average='weighted'):.4f}")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

### Claude Code — Task 2

**Score: 12/12** — Pipeline with ColumnTransformer (StandardScaler for numeric, OneHotEncoder for Embarked). All preprocessing inside Pipeline, no leakage. Metrics: Accuracy 0.8045, Precision 0.8034, Recall 0.8045, F1 0.8007.

In [ ]:
"""
Titanic Survival Prediction - Logistic Regression Classifier
Trained on titanic_clean.csv with stratified 80/20 train/test split.
RANDOM_STATE = 42 throughout.
"""

import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

RANDOM_STATE = 42

# ---------------------------------------------------------------------------
# 1. Load data
# ---------------------------------------------------------------------------
data_path = Path(__file__).parent.parent / "Claude Code" / "titanic_clean.csv"
df = pd.read_csv(data_path)

# ---------------------------------------------------------------------------
# 2. Split into features and target
# ---------------------------------------------------------------------------
X = df.drop(columns=["Survived"])
y = df["Survived"]

# ---------------------------------------------------------------------------
# 3. Stratified 80/20 train/test split
# ---------------------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

# ---------------------------------------------------------------------------
# 4. Build preprocessing + classifier Pipeline
#    Preprocessing happens inside the pipeline, fitted on training data only.
# ---------------------------------------------------------------------------
numeric_features = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare"]
categorical_features = ["Embarked"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    ]
)

# Fit on training data only
pipeline.fit(X_train, y_train)

# ---------------------------------------------------------------------------
# 5. Evaluate on test set
# ---------------------------------------------------------------------------
y_pred = pipeline.predict(X_test)

accuracy  = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average="weighted")
recall    = recall_score(y_test, y_pred, average="weighted")
f1        = f1_score(y_test, y_pred, average="weighted")

print("=" * 40)
print("  Logistic Regression — Test Results")
print("=" * 40)
print(f"  Accuracy  : {accuracy:.4f}")
print(f"  Precision : {precision:.4f}  (weighted)")
print(f"  Recall    : {recall:.4f}  (weighted)")
print(f"  F1 Score  : {f1:.4f}  (weighted)")

# ---------------------------------------------------------------------------
# 6. Confusion matrix
# ---------------------------------------------------------------------------
cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:")
print(f"  Labels: [Not Survived (0), Survived (1)]")
print(f"\n  {'':>15} Predicted 0  Predicted 1")
print(f"  {'Actual 0':>15}  {cm[0, 0]:^11}  {cm[0, 1]:^11}")
print(f"  {'Actual 1':>15}  {cm[1, 0]:^11}  {cm[1, 1]:^11}")
print("=" * 40)


### Codex — Task 2

**Score: 12/12** — Same Pipeline architecture but with dynamic `select_dtypes` for feature detection. Metrics identical to Claude Code at 4dp, confirming deterministic pipeline.

In [ ]:
from __future__ import annotations

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


RANDOM_STATE = 42
INPUT_PATH = "titanic_clean.csv"
TARGET_COLUMN = "Survived"


def main() -> None:
    df = pd.read_csv(INPUT_PATH)

    X = df.drop(columns=TARGET_COLUMN)
    y = df[TARGET_COLUMN]

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        stratify=y,
        random_state=RANDOM_STATE,
    )

    categorical_features = X_train.select_dtypes(
        include=["object", "category", "string"]
    ).columns.tolist()
    numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()

    preprocessor = ColumnTransformer(
        transformers=[
            ("numeric", StandardScaler(), numeric_features),
            ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ]
    )

    model = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            (
                "classifier",
                LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
            ),
        ]
    )

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
    print(f"Precision (weighted): {precision_score(y_test, y_pred, average='weighted'):.4f}")
    print(f"Recall (weighted): {recall_score(y_test, y_pred, average='weighted'):.4f}")
    print(f"F1 score (weighted): {f1_score(y_test, y_pred, average='weighted'):.4f}")
    print("Confusion matrix:")
    print(confusion_matrix(y_test, y_pred))


if __name__ == "__main__":
    main()


---
## Task 3: Detecting and Fixing Data Leakage

**Specification:** Given `titanic_pipeline_v1.py` (StandardScaler fitted before train/test split), identify the leakage, fix it, run both versions, and compare metrics.

**Success criteria:** Leakage mechanism identified and explained, corrected script saved, both sets of metrics reported.

### The Broken Script (given to all tools)

In [ ]:


import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score,recall_score, f1_score
RANDOM_STATE = 42
df = pd.read_csv('titanic_clean.csv')
X = df.drop('Survived', axis=1)
y = df['Survived']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X) 
X_train, X_test, y_train, y_test = train_test_split(
X_scaled, y, test_size=0.2, stratify=y,
random_state=RANDOM_STATE
)
model = LogisticRegression(max_iter=1000,
random_state=RANDOM_STATE)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred,
average='weighted'):.4f}")
print(f"Recall: {recall_score(y_test, y_pred,
average='weighted'):.4f}")
print(f"F1: {f1_score(y_test, y_pred,
average='weighted'):.4f}")

### Copilot — Task 3

**Score: 5/12** — Correctly identified the leakage mechanism and produced a fix using `scaler.fit(X_train)` / `scaler.transform(X_test)`. However, it could not run either version to produce measured metrics (no execution capability). Estimated that the leaky version would score higher — empirically incorrect at this dataset size.

In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

RANDOM_STATE = 42

df = pd.read_csv("titanic_clean.csv")
X = df.drop("Survived", axis=1)
y = df["Survived"]

# Split before scaling to avoid leakage
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred, average='weighted'):.4f}")
print(f"Recall: {recall_score(y_test, y_pred, average='weighted'):.4f}")
print(f"F1: {f1_score(y_test, y_pred, average='weighted'):.4f}")

### Claude Code — Task 3

**Score: 6/12** — Identified the leakage mechanism with detailed inline annotations. Also detected the secondary Embarked string issue. However, no separate fixed script was saved to disk — only the annotated leaky version was preserved. The scoring penalty reflects the absence of a deliverable, not understanding.

In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

RANDOM_STATE = 42

df = pd.read_csv('titanic_clean.csv')
# Encode Embarked (left as string by preprocess.py)
df = pd.get_dummies(df, columns=['Embarked'], drop_first=True)

X = df.drop('Survived', axis=1)
y = df['Survived']

# BUG: scaler is fit on ALL data (including future test rows) before the split
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)   # <-- data leakage here
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, stratify=y,
    random_state=RANDOM_STATE
)

model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred, average='weighted'):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred, average='weighted'):.4f}")
print(f"F1:        {f1_score(y_test, y_pred, average='weighted'):.4f}")


### Codex — Task 3

**Score: 12/12** — Full Pipeline restructure from raw `titanic.csv`: ColumnTransformer with separate sub-pipelines (SimpleImputer+StandardScaler for numeric, SimpleImputer+OneHotEncoder for categorical), all fitted after split. Eliminates leakage by design. Metrics: Accuracy 0.8045, F1 0.8007.

In [ ]:
from __future__ import annotations

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


RANDOM_STATE = 42
INPUT_PATH = "titanic.csv"
TARGET_COLUMN = "Survived"


def main() -> None:
    df = pd.read_csv(INPUT_PATH)

    columns_to_drop = ["PassengerId", "Name", "Ticket", "Cabin"]
    existing_columns_to_drop = [
        column for column in columns_to_drop if column in df.columns
    ]

    X = df.drop(columns=[TARGET_COLUMN, *existing_columns_to_drop])
    y = df[TARGET_COLUMN]

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        stratify=y,
        random_state=RANDOM_STATE,
    )

    categorical_features = X_train.select_dtypes(
        include=["object", "category", "string"]
    ).columns.tolist()
    numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()

    numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore")),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("numeric", numeric_transformer, numeric_features),
            ("categorical", categorical_transformer, categorical_features),
        ]
    )

    model = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            (
                "classifier",
                LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
            ),
        ]
    )

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
    print(f"Precision (weighted): {precision_score(y_test, y_pred, average='weighted'):.4f}")
    print(f"Recall (weighted): {recall_score(y_test, y_pred, average='weighted'):.4f}")
    print(f"F1 score (weighted): {f1_score(y_test, y_pred, average='weighted'):.4f}")


if __name__ == "__main__":
    main()


---
## Task 4: Debugging a Broken Pipeline

**Specification:** Given `titanic_pipeline_v2.py` with two planted bugs (shuffle=False, predict on X_train instead of X_test), identify both, explain consequences, fix, and report corrected metrics.

**Success criteria:** Both bugs found, consequences explained, corrected script runs, metrics reported.

### The Broken Script (given to all tools)

In [ ]:


import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score,recall_score, f1_score
RANDOM_STATE = 42
df = pd.read_csv('titanic_clean.csv')
X = df.drop('Survived', axis=1)
y = df['Survived']


X_train, X_test, y_train, y_test = train_test_split(
X, y, test_size=0.2, shuffle=False 
)
model = LogisticRegression(max_iter=1000,
random_state=RANDOM_STATE)
model.fit(X_train, y_train)


y_pred = model.predict(X_train) 
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred,
average='weighted'):.4f}")
print(f"Recall: {recall_score(y_test, y_pred,
average='weighted'):.4f}")
print(f"F1: {f1_score(y_test, y_pred,
average='weighted'):.4f}")

### Copilot — Task 4

**Score: 9/12** — Identified both bugs with consequence-level explanations on first read. Fixed script still crashed at runtime due to Embarked string issue (same T1 cascade). Reviewer-verified metrics: Accuracy ≈ 0.81.

In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

RANDOM_STATE = 42

df = pd.read_csv('titanic_clean.csv')

X = df.drop('Survived', axis=1)
y = df['Survived']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    shuffle=True,
    random_state=RANDOM_STATE
)

model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred, average='weighted'):.4f}")
print(f"Recall: {recall_score(y_test, y_pred, average='weighted'):.4f}")
print(f"F1: {f1_score(y_test, y_pred, average='weighted'):.4f}")

### Claude Code — Task 4

**Score: 12/12** — Found both bugs with detailed inline comments explaining consequences. Added `pd.get_dummies` for Embarked encoding. Script ran end-to-end. Metrics: Accuracy 0.8101, Precision 0.8092, Recall 0.8101, F1 0.8092.

In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

RANDOM_STATE = 42

df = pd.read_csv('titanic_clean.csv')
# Encode any remaining categorical columns so LogisticRegression can accept them
df = pd.get_dummies(df, columns=['Embarked'], drop_first=True)
X = df.drop('Survived', axis=1)
y = df['Survived']

# Bug 1 fixed: shuffle=False removed (default shuffle=True ensures a
# representative, unbiased split instead of preserving the original row order,
# which in the Titanic dataset is correlated with passenger class / survival).
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
model.fit(X_train, y_train)

# Bug 2 fixed: predict on X_test (not X_train).
# The original code called model.predict(X_train), producing predictions whose
# length matches the training set (~800) but compared them against y_test
# (~200), causing a ValueError and — even conceptually — measuring training
# predictions against test labels, which is invalid.
y_pred = model.predict(X_test)

print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred, average='weighted'):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred, average='weighted'):.4f}")
print(f"F1:        {f1_score(y_test, y_pred, average='weighted'):.4f}")


### Codex — Task 4

**Score: 11/12** — Found both bugs plus the Embarked encoding issue. Replaced flat script with Pipeline architecture. Minor deviation: used binary averaging instead of weighted for precision/recall/F1. Metrics: Accuracy 0.8045, Precision 0.7931 (binary), Recall 0.6667 (binary), F1 0.7244 (binary).

In [ ]:
from __future__ import annotations

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


RANDOM_STATE = 42
INPUT_PATH = "titanic_clean.csv"
TARGET_COLUMN = "Survived"


def main() -> None:
    df = pd.read_csv(INPUT_PATH)

    X = df.drop(columns=TARGET_COLUMN)
    y = df[TARGET_COLUMN]

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=RANDOM_STATE,
        stratify=y,
    )

    categorical_features = X_train.select_dtypes(
        include=["object", "category", "string"]
    ).columns.tolist()
    numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()

    preprocessor = ColumnTransformer(
        transformers=[
            ("numeric", StandardScaler(), numeric_features),
            ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ]
    )

    model = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("classifier", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
        ]
    )

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
    print(f"Precision: {precision_score(y_test, y_pred):.4f}")
    print(f"Recall: {recall_score(y_test, y_pred):.4f}")
    print(f"F1: {f1_score(y_test, y_pred):.4f}")


if __name__ == "__main__":
    main()


---
## Task 5: EDA and Insight Generation

**Specification:** Load `titanic_clean.csv`, print summary statistics and `.describe()`, compute survival rates by Sex and Pclass, generate and save four plots (survival count bar chart, age distribution histogram by survival, survival rate by sex, correlation heatmap).

**Success criteria:** Script runs, all 4 PNGs saved, survival rates correct (female ~74%, male ~19%; 1st class ~63%, 3rd class ~24%), no hallucinated columns.

### Copilot — Task 5

**Score: 5/12** — Produced code for all 4 plots, but `df.corr()` crashed on the string Embarked column (inherited from T1's silent corruption), so `eda_correlation_heatmap.png` was never saved. Only 3/4 PNGs on disk. This is the cascading failure pattern from T1 repeating in a new context — the fifth consecutive task affected by the same upstream defect.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_STATE = 42
df = pd.read_csv('titanic_clean.csv')

print("Shape:", df.shape)
print("\nColumn names and dtypes:")
print(df.dtypes)
print("\nDescribe:")
print(df.describe())

print("\nSurvival rates by Sex:")
survival_by_sex = df.groupby('Sex')['Survived'].mean() * 100
print(f"Male: {survival_by_sex[0]:.1f}%")
print(f"Female: {survival_by_sex[1]:.1f}%")

print("\nSurvival rates by Pclass:")
survival_by_pclass = df.groupby('Pclass')['Survived'].mean() * 100
for pclass in sorted(survival_by_pclass.index):
    print(f"Class {pclass}: {survival_by_pclass[pclass]:.1f}%")

# a. Survival count bar chart
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='Survived')
plt.title('Survival Count')
plt.savefig('eda_survival_count.png')
print("Saved eda_survival_count.png")

# b. Age distribution histogram
plt.figure(figsize=(8, 6))
sns.histplot(data=df, x='Age', bins=20, hue='Survived', multiple='stack')
plt.title('Age Distribution by Survival')
plt.savefig('eda_age_distribution.png')
print("Saved eda_age_distribution.png")

# c. Survival rate by Sex
plt.figure(figsize=(6, 4))
sns.barplot(data=df, x='Sex', y='Survived', estimator=lambda x: sum(x)/len(x)*100)
plt.ylabel('Survival Rate (%)')
plt.title('Survival Rate by Sex')
plt.savefig('eda_survival_by_sex.png')
print("Saved eda_survival_by_sex.png")

# d. Correlation heatmap
plt.figure(figsize=(8, 6))
corr = df.corr()  # CRASHES: ValueError — Embarked is still a string
sns.heatmap(corr, annot=True, cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.savefig('eda_correlation_heatmap.png')
print("Saved eda_correlation_heatmap.png")

### Claude Code — Task 5

**Score: 12/12** — All 4 plots saved, `Agg` backend, `select_dtypes(include='number')` for heatmap. Survival rates correct: male 18.9%, female 74.2%; 1st class 63.0%, 3rd class 24.2%.

In [ ]:
import matplotlib
matplotlib.use('Agg')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_STATE = 42

# ── 1. Load and describe ──────────────────────────────────────────────────────
df = pd.read_csv('titanic_clean.csv')

print("=== Shape ===")
print(df.shape)

print("\n=== Column names and dtypes ===")
print(df.dtypes)

print("\n=== describe() ===")
print(df.describe())

# ── 2. Survival rates ─────────────────────────────────────────────────────────
print("\n=== Survival rate by Sex ===")
sex_rate = df.groupby('Sex')['Survived'].mean() * 100
for sex, rate in sex_rate.items():
    print(f"  {sex}: {rate:.1f}%")

print("\n=== Survival rate by Pclass ===")
pclass_rate = df.groupby('Pclass')['Survived'].mean() * 100
for pclass, rate in pclass_rate.items():
    print(f"  Class {pclass}: {rate:.1f}%")

# ── 3a. Survival count bar chart ──────────────────────────────────────────────
fig, ax = plt.subplots()
df['Survived'].value_counts().sort_index().plot(kind='bar', ax=ax, color=['steelblue', 'salmon'])
ax.set_title('Survival Counts')
ax.set_xlabel('Survived (0 = No, 1 = Yes)')
ax.set_ylabel('Count')
ax.set_xticklabels(['0', '1'], rotation=0)
plt.tight_layout()
plt.savefig('eda_survival_count.png')
plt.close()
print("Saved eda_survival_count.png")

# ── 3b. Age distribution histogram coloured by Survived ───────────────────────
fig, ax = plt.subplots()
for survived, grp in df.groupby('Survived'):
    label = 'Survived' if survived == 1 else 'Not Survived'
    ax.hist(grp['Age'].dropna(), bins=20, alpha=0.6, label=label)
ax.set_title('Age Distribution by Survival')
ax.set_xlabel('Age')
ax.set_ylabel('Count')
ax.legend()
plt.tight_layout()
plt.savefig('eda_age_distribution.png')
plt.close()
print("Saved eda_age_distribution.png")

# ── 3c. Survival rate by Sex grouped bar chart ────────────────────────────────
fig, ax = plt.subplots()
sex_survival = df.groupby('Sex')['Survived'].mean() * 100
sex_survival.plot(kind='bar', ax=ax, color=['steelblue', 'salmon'])
ax.set_title('Survival Rate by Sex')
ax.set_xlabel('Sex')
ax.set_ylabel('Survival Rate (%)')
ax.set_xticklabels(sex_survival.index, rotation=0)
plt.tight_layout()
plt.savefig('eda_survival_by_sex.png')
plt.close()
print("Saved eda_survival_by_sex.png")

# ── 3d. Correlation heatmap ───────────────────────────────────────────────────
numeric_df = df.select_dtypes(include='number')
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(numeric_df.corr(), annot=True, fmt='.2f', cmap='coolwarm', ax=ax)
ax.set_title('Correlation Heatmap of Numeric Features')
plt.tight_layout()
plt.savefig('eda_correlation_heatmap.png')
plt.close()
print("Saved eda_correlation_heatmap.png")


### Codex — Task 5

**Score: 12/12** — All 4 plots saved at 150 DPI. Most polished: modular functions, type hints, `build_sex_labels()` maps binary Sex back to male/female for display, percentage annotations on bar charts. Best code quality of the three.

In [ ]:
import matplotlib

matplotlib.use("Agg")

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


RANDOM_STATE = 42
CSV_PATH = Path("titanic_clean.csv")


def format_percentage(series: pd.Series) -> pd.Series:
    return (series * 100).round(1)


def build_sex_labels(sex_series: pd.Series) -> pd.Series:
    sex_map = {0: "male", 1: "female", "0": "male", "1": "female"}
    return sex_series.map(sex_map).fillna(sex_series.astype(str))


def print_dataset_summary(df: pd.DataFrame) -> None:
    print("Dataset shape:")
    print(df.shape)
    print("\nColumns:")
    print(df.columns.tolist())
    print("\nData types:")
    print(df.dtypes)
    print("\nDescribe:")
    print(df.describe(include="all"))


def print_survival_rates(df: pd.DataFrame) -> None:
    sex_labels = build_sex_labels(df["Sex"])
    sex_rates = format_percentage(df.groupby(sex_labels)["Survived"].mean())
    pclass_rates = format_percentage(df.groupby("Pclass")["Survived"].mean())

    print("\nSurvival rate by Sex:")
    for label in ["male", "female"]:
        if label in sex_rates.index:
            print(f"{label}: {sex_rates[label]:.1f}%")

    print("\nSurvival rate by Pclass:")
    class_labels = {1: "1st", 2: "2nd", 3: "3rd"}
    for pclass in [1, 2, 3]:
        if pclass in pclass_rates.index:
            print(f"{class_labels[pclass]}: {pclass_rates[pclass]:.1f}%")


def save_survival_count_plot(df: pd.DataFrame) -> None:
    counts = df["Survived"].value_counts().sort_index()

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(counts.index.astype(str), counts.values, color=["#d95f02", "#1b9e77"])
    ax.set_title("Survival Counts")
    ax.set_xlabel("Survived")
    ax.set_ylabel("Count")
    fig.tight_layout()
    fig.savefig("eda_survival_count.png", dpi=150)
    plt.close(fig)
    print("Saved eda_survival_count.png")


def save_age_distribution_plot(df: pd.DataFrame) -> None:
    survived_groups = {
        0: ("Did not survive", "#d95f02"),
        1: ("Survived", "#1b9e77"),
    }

    fig, ax = plt.subplots(figsize=(8, 5))
    for survived, (label, color) in survived_groups.items():
        ages = df.loc[df["Survived"] == survived, "Age"].dropna()
        ax.hist(ages, bins=20, alpha=0.6, label=label, color=color, edgecolor="black")

    ax.set_title("Age Distribution by Survival")
    ax.set_xlabel("Age")
    ax.set_ylabel("Count")
    ax.legend()
    fig.tight_layout()
    fig.savefig("eda_age_distribution.png", dpi=150)
    plt.close(fig)
    print("Saved eda_age_distribution.png")


def save_survival_by_sex_plot(df: pd.DataFrame) -> None:
    sex_labels = build_sex_labels(df["Sex"])
    survival_rates = df.assign(SexLabel=sex_labels).groupby("SexLabel")["Survived"].mean()
    ordered_index = [label for label in ["male", "female"] if label in survival_rates.index]
    survival_rates = survival_rates.reindex(ordered_index)

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(survival_rates.index, survival_rates.values * 100, color=["#7570b3", "#e7298a"][: len(survival_rates)])
    ax.set_title("Survival Rate by Sex")
    ax.set_xlabel("Sex")
    ax.set_ylabel("Survival Rate (%)")
    ax.set_ylim(0, 100)

    for idx, value in enumerate(survival_rates.values * 100):
        ax.text(idx, value + 1, f"{value:.1f}%", ha="center")

    fig.tight_layout()
    fig.savefig("eda_survival_by_sex.png", dpi=150)
    plt.close(fig)
    print("Saved eda_survival_by_sex.png")


def save_correlation_heatmap(df: pd.DataFrame) -> None:
    numeric_df = df.select_dtypes(include=[np.number])
    corr = numeric_df.corr()

    fig, ax = plt.subplots(figsize=(8, 6))
    heatmap = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
    ax.set_title("Correlation Heatmap")
    ax.set_xticks(range(len(corr.columns)))
    ax.set_yticks(range(len(corr.index)))
    ax.set_xticklabels(corr.columns, rotation=45, ha="right")
    ax.set_yticklabels(corr.index)

    for i in range(len(corr.index)):
        for j in range(len(corr.columns)):
            ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", color="black", fontsize=8)

    fig.colorbar(heatmap, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    fig.savefig("eda_correlation_heatmap.png", dpi=150)
    plt.close(fig)
    print("Saved eda_correlation_heatmap.png")


def main() -> None:
    np.random.seed(RANDOM_STATE)
    df = pd.read_csv(CSV_PATH)

    print_dataset_summary(df)
    print_survival_rates(df)
    save_survival_count_plot(df)
    save_age_distribution_plot(df)
    save_survival_by_sex_plot(df)
    save_correlation_heatmap(df)


if __name__ == "__main__":
    main()


---
## Results Summary

In [ ]:
import pandas as pd

# Task scores (out of 12 per task)
data = {
    'Task': ['T1: Ingestion', 'T2: Baseline Model', 'T3: Leakage Detection', 'T4: Debugging', 'T5: EDA', 'TOTAL'],
    'Copilot': [4, 6, 5, 9, 5, 29],
    'Claude Code': [12, 12, 6, 12, 12, 54],
    'Codex': [12, 12, 12, 11, 12, 59]
}
scores = pd.DataFrame(data).set_index('Task')
print("=" * 55)
print("  BENCHMARK RESULTS — Score per Task (max 12)")
print("=" * 55)
print(scores.to_string())
print()
print(f"Overall max possible: 60 (5 tasks x 12)")

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

tasks = ['T1:\nIngestion', 'T2:\nBaseline', 'T3:\nLeakage', 'T4:\nDebugging', 'T5:\nEDA']
copilot =    [4, 6, 5, 9, 5]
claude_code = [12, 12, 6, 12, 12]
codex =      [12, 12, 12, 11, 12]

x = np.arange(len(tasks))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width, copilot, width, label='Copilot', color='#e74c3c', alpha=0.85)
bars2 = ax.bar(x, claude_code, width, label='Claude Code', color='#3498db', alpha=0.85)
bars3 = ax.bar(x + width, codex, width, label='Codex', color='#2ecc71', alpha=0.85)

ax.set_ylabel('Score (max 12)')
ax.set_title('Benchmark Scores by Task and Tool')
ax.set_xticks(x)
ax.set_xticklabels(tasks)
ax.set_ylim(0, 14)
ax.legend()
ax.axhline(y=12, color='gray', linestyle='--', alpha=0.3)

for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{int(height)}', xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('benchmark_comparison.png', dpi=150)
plt.close()
print("Saved benchmark_comparison.png")

---
## Key Findings

**1. Execution capability is the defining divide.** Copilot Chat cannot run code, observe failures, or self-correct. This single constraint explains the majority of its scoring gap (29/60 vs Claude Code 54/60 and Codex 59/60). Tasks requiring validation of outputs (T1, T5) or measured metrics (T3) are systematically disadvantaged.

**2. Cascading failures amplify upstream defects — but only for tools that cannot self-verify.** Copilot's T1 `inplace` bug was undetectable without execution, and it propagated into T2 (ValueError crash) and T5 (correlation heatmap crash). Claude Code and Codex both validated their T1 output before proceeding, breaking the cascade.

**3. Static code review is broadly reliable; producing verified fixes is harder.** All three tools correctly identified planted bugs in T3 and T4. But delivering a complete, runnable, saved fix required execution capability. Copilot's T4 fix crashed at runtime; Claude Code's T3 fix was never saved to disk; only Codex delivered verified fixes for both.

**4. Code quality scales with architectural sophistication.** Codex consistently produced the most structured code (type hints, modular functions, dynamic feature detection, `main()` entry points). Claude Code was clean and functional. Copilot's code was structurally reasonable but untested against its own outputs.